In [ ]:
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)
import trajectory_data
import matplotlib.pyplot as plt
from nocode_robot_programming.state_decision.dataloader import TrajectoryDataset
from nocode_robot_programming.state_decision.dino_model import DINOStateDecider
from nocode_robot_programming.state_decision.SIFT_model import StateDeciderSIFT
from nocode_robot_programming.state_decision.AEGP_model import AEGP
from nocode_robot_programming.state_decision.state_decider import StateDeciderBase
from nocode_robot_programming.state_decision.utils import Filename
from gesture_detector.utils import pretty_confusion_matrix
import torch
import numpy as np

from trajectory_data.skill_visualizer import play_video
from nocode_robot_programming.state_decision.utils import add_tag
from nocode_robot_programming.state_decision.dataloader import ImageDatasetView, saved_img_processing


In [ ]:
dataset = TrajectoryDataset(trajectory_data.package_path)

In [ ]:
task_name = "d1_peg_pick"

In [ ]:
d = dataset.get_image_dataset(dataset.tasks[task_name]['names'])

In [ ]:
file_names = dataset.tasks[task_name]['names']

In [ ]:
def option2_trials_are_test_data(dataset, file_names: list[str], at=slice(35,85)) -> ImageDatasetView:
    X = torch.tensor([])
    Xt = torch.tensor([])
    y_int = torch.tensor([], dtype=torch.int)
    y_names = []

    tags = []
    for file in file_names:
        t = dataset[file]['tag']
        if t not in tags:
            tags.append(t)

    X_parts, Xt_parts, y_int_parts, y_name_parts = [], [], [], []

    start = at.start if at.start is not None else -float("inf")
    stop  = at.stop  if at.stop  is not None else  float("inf")

    for file in file_names:
        f = Filename(file)
        idx = dataset.files.index(f"{dataset.dir}/{file}.npz")
        imgs = saved_img_processing(dataset[idx]['img'].squeeze()).squeeze()   # (nsamples, H, W)
        nsamples = imgs.shape[0]
        
        tag = dataset[file]['tag']
        i = tags.index(tag)
        xt = torch.arange(f.offset, f.offset + nsamples)  # shape (nsamples,)
        
        mask = (xt >= start) & (xt < stop)                # safe, no OOB
        if mask.any():
            imgs_sub = imgs[mask]
            xt_sub   = xt[mask]

            y_int_sub   = torch.full((imgs_sub.shape[0],), i, dtype=torch.int)
            y_names_sub = [tag] * imgs_sub.shape[0]

            X_parts.append(imgs_sub)
            Xt_parts.append(xt_sub)
            y_int_parts.append(y_int_sub)
            y_name_parts.extend(y_names_sub)

    X  = torch.cat(X_parts, dim=0)             # (total_samples, H, W)
    Xt = torch.cat(Xt_parts, dim=0)            # (total_samples,)
    y_int = torch.cat(y_int_parts, dim=0)      # (total_samples,)
    y_names = y_name_parts

    return ImageDatasetView(X=X, Xt=Xt, y_int=y_int, y_names=y_names, y_cls=tags)

In [ ]:
dataset.play_video(idx='d1_peg_pick_trial_3')

In [ ]:
d_train.play_video(fps=10)

In [ ]:
d_train = option2_trials_are_test_data(dataset, ['d1_peg_pick_trial_3', 'd1_peg_pick_branch_at_49'], at=slice(49-30, 49+30))


In [ ]:
d_train.show_image(80)

In [ ]:
# for task_name in dataset.tasks.keys():
task_name = 'd1_peg_pick'
train_names = [n for n in dataset.tasks[task_name]['names'] if "trial" not in n or n == 'd1_peg_pick_trial_3']
test_names = [n for n in dataset.tasks[task_name]['names'] if "trial" in n and n != 'd1_peg_pick_trial_3']

d_train = option2_trials_are_test_data(dataset, train_names, at=slice(49-20, 49+20))
d_test = option2_trials_are_test_data(dataset, test_names, at=slice(49-20, 49+20))
for decider in [
        DINOStateDecider(),
        # StateDeciderSIFT(),
        # AEGP(),
    ]:
    decider.train(d_train.X, d_train.y_int, d_train.y_cls)
    y_pred = decider.predict_many(d_test.X)
    
    pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, figsize=(len(decider.y_cls)+3,len(decider.y_cls)+3), name=decider.__class__.__name__)

# TODO

- This dataset doesn't have a huge accuracy, what should I do.
I'm not very nible at observing the dataset, need a few more covenience functions.